In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the final dataset
df = pd.read_csv('../data/annotated/final_labeled_fintech_data.csv')

# Verify we still have our 572 records
print(f"Quality Check initialized for {len(df)} records.")

Quality Check initialized for 572 records.


In [3]:
# Flag reviews with fewer than 3 words
df['word_count'] = df['cleaned_review'].apply(lambda x: len(str(x).split()))
short_reviews = df[df['word_count'] <= 2]

print(f"Found {len(short_reviews)} ultra-short reviews.")
short_reviews[['cleaned_review', 'sentiment']].head(10)

Found 117 ultra-short reviews.


,cleaned_review,sentiment
0,very good,Positive
2,good app,Positive
5,excellent,Positive
6,awesome,Positive
8,good,Positive
15,wonderful,Positive
18,extraordinary,Positive
22,top notch,Positive
30,satisfactory,Positive
34,very sharp,Positive


In [4]:
# Define 'Red Flag' words for Negative sentiment
neg_keywords = ['failed', 'stole', 'worst', 'scam', 'useless', 'slow', 'error']

# Find reviews labeled 'Positive' that contain 'Negative' keywords
mismatch = df[(df['sentiment'] == 'Positive') & 
              (df['cleaned_review'].str.contains('|'.join(neg_keywords), case=False))]

print(f"Potential Mismatches found: {len(mismatch)}")
mismatch[['cleaned_review', 'sentiment']]

Potential Mismatches found: 0


,cleaned_review,sentiment


In [5]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

def get_vader_label(text):
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05: return 'Positive'
    if score <= -0.05: return 'Negative'
    return 'Neutral'

# Run VADER on your data
df['vader_label'] = df['cleaned_review'].apply(get_vader_label)

# Calculate Agreement
agreement = (df['sentiment'] == df['vader_label']).mean()
print(f"Human-to-Algorithm Agreement Rate: {agreement:.2%}")

# Show the "Conflicts"
conflicts = df[df['sentiment'] != df['vader_label']]
conflicts[['cleaned_review', 'sentiment', 'vader_label']].head()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Cobekpa\AppData\Roaming\nltk_data...


Human-to-Algorithm Agreement Rate: 81.99%


,cleaned_review,sentiment,vader_label
10,you guys keep logging me out i have to reset p...,Negative,Neutral
14,kuda bank is incredible app to bank with,Positive,Neutral
16,"i rate this app a1, don't know about anyone else",Positive,Neutral
17,durable and reliable,Positive,Neutral
18,extraordinary,Positive,Neutral
